# Fase 1 · Paso 1: auditoría del piloto DICOM

Esta libreta solo orquesta el código versionado del repositorio. Reconciliará diez adquisiciones únicas, guardará evidencia privada en Drive y no entrenará modelos ni abrirá una prueba reservada.

## 1. Montar Drive y localizar la raíz privada

La ruta se obtiene del secreto `KNEE_DATA_ROOT`; no se almacena en la libreta.

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT antes de continuar.')
os.environ['KNEE_DATA_ROOT'] = private_root
if not Path(private_root).is_dir():
    raise FileNotFoundError('La raíz privada configurada no está disponible.')
print('Drive montado y raíz privada localizada.')

## 2. Obtener una revisión identificable del repositorio

In [ ]:
import shutil
import subprocess

REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_fase1_paso1')

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('La ruta de trabajo existe y no es un clon Git; usa un entorno nuevo.')
    dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('El clon temporal contiene cambios. Restablece el entorno de ejecución.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print('Revisión preparada:', commit)

## 3. Instalar y comprobar el código

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)
if subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout:
    raise RuntimeError('Las comprobaciones modificaron el repositorio temporal.')
print('Pruebas automáticas completadas.')

## 4. Auditar el piloto canónico de Drive

In [ ]:
config_path = REPO_DIR / 'configs' / 'pilot_audit.example.json'
subprocess.run([sys.executable, '-m', 'knee.dicom_audit', '--config', str(config_path)], cwd=REPO_DIR, check=True)

## 5. Verificar el cierre y registrar la ejecución

In [ ]:
import json
from datetime import datetime, timezone

output_dir = Path(private_root) / 'outputs' / 'auditorias' / 'fase_1' / 'paso_1_piloto'
summary_path = output_dir / 'auditoria_piloto_publica.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
if summary.get('status') != 'ok' or summary.get('selected_unique_acquisitions') != 10:
    raise RuntimeError('El piloto no cumple el criterio de diez adquisiciones únicas.')
record = {
    'phase': 1,
    'step': 1,
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': commit,
    'audit_status': summary['status'],
    'selected_unique_acquisitions': summary['selected_unique_acquisitions'],
    'training_executed': False,
    'reserved_test_opened': False,
}
(output_dir / 'registro_ejecucion.json').write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Fase 1, paso 1 ejecutado correctamente. Evidencias privadas guardadas en Drive.', record)